# ThermoRG Phase A v2 — CIFAR-10 D-Scaling
## Validates ThermoRG v3: β∝J_topo, α∝J_topo² on real CIFAR-10 data

**Hypotheses:**
- H1: β̂ ∝ J_topo (Pearson r > 0.7, p < 0.05)
- H2: α̂ ∝ J_topo² (Pearson r > 0.7, p < 0.05)

**12 architectures**: ThermoNet (width/depth families), ResNet-18/34, VGG-11/13
**D ∈ {2K, 5K, 10K, 25K, 50K}** × 2 seeds × 50 epochs

---


In [ ]:
import os, sys, subprocess
from kaggle_secrets import UserSecretsClient

## ▶ufe0f Step 1: Clone Code from GitHub

In [ ]:
%cd /kaggle/working
!rm -rf ThermoRG-NN
user_secrets = UserSecretsClient()
gh_token = user_secrets.get_secret("gh_token")
repo_url = f"https://{gh_token}@github.com/xliu203/ThermoRG-NN.git"
!git clone {repo_url} --branch develop
%cd /kaggle/working/ThermoRG-NN
!git config --global user.email "xliu203@asu.edu"
!git config --global user.name "Leo Liu"
print("Cloned develop branch")

## ▶ufe0f Step 2: Install Dependencies

In [ ]:
!pip install -q torch torchvision scipy numpy matplotlib
sys.path.insert(0, "/kaggle/working/ThermoRG-NN")
sys.path.insert(0, "/kaggle/working/ThermoRG-NN/experiments/phase_a")
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## ▶ufe0f Step 3: Smoke Test — Forward + Backward All 12 Architectures

In [ ]:
import torch, torch.nn as nn
import phase_a_dscaling as p

x = torch.randn(2, 3, 32, 32)
y = torch.randint(0, 10, (2,))

ARCHITECTURES = [
    {"Name": "TN-W8",  "arch": "TN3",  "width_mult": 0.25},
    {"Name": "TN-W16", "arch": "TN3",  "width_mult": 0.5},
    {"Name": "TN-W32", "arch": "TN3",  "width_mult": 1.0},
    {"Name": "TN-W64", "arch": "TN5",  "width_mult": 0.5},
    {"Name": "TN-L3",  "arch": "TN3",  "width_mult": 1.0},
    {"Name": "TN-L5",  "arch": "TN5",  "width_mult": 1.0},
    {"Name": "TN-L7",  "arch": "TN7",  "width_mult": 1.0},
    {"Name": "TN-L9",  "arch": "TN9",  "width_mult": 1.0},
    {"Name": "ResNet-18", "arch": "R18",  "width_mult": 1.0},
    {"Name": "ResNet-34", "arch": "R34",  "width_mult": 1.0},
    {"Name": "VGG-11",    "arch": "VGG11","width_mult": 1.0},
    {"Name": "VGG-13",    "arch": "VGG13","width_mult": 1.0},
]

all_pass = True
for cfg in ARCHITECTURES:
    try:
        m = p.get_arch_model(cfg["arch"], cfg.get("width_mult", 1.0))
        out = m(x)
        loss = nn.CrossEntropyLoss()(out, y)
        loss.backward()
        params_M = sum(p.numel() for p in m.parameters()) / 1e6
        has_grad = all(p.grad is not None for p in m.parameters() if p.requires_grad)
        status = "✓" if has_grad else "✗"
        if not has_grad: all_pass = False
        print(f"  [{cfg['Name']:12s}] {params_M:.2f}M  Fwd+Bwd: {status}")
    except Exception as e:
        print(f"  [{cfg['Name']:12s}] ERROR: {e}")
        all_pass = False

print(f"\n{'ALL 12 PASS ✓' if all_pass else 'SOME FAILED ✗'}")

## ▶ufe0f Step 4: J_topo v3 Smoke Test

In [ ]:
conv_w = torch.nn.Conv2d(3, 64, 3, padding=1).weight
lin_w  = torch.nn.Linear(64, 10).weight
J, etas = p.compute_J_topo([conv_w, lin_w])
print(f"J_topo = {J:.4f}")
print(f"eta_vals = {[f'{e:.3f}' for e in etas]}")
print(f"J ∈ (0,1]: {'✓' if 0 < J <= 1 else '✗'}")

## ▶ufe0f Step 5: Run Phase A v2 Training

**Warning**: ~4-8 hours on T4×2. Checkpoint resume is enabled — interruption is safe.

In [ ]:
try:
    print("Starting Phase A v2 training...")
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
    print("Architecture count: 12")
    print("D values: {2K, 5K, 10K, 25K, 50K}")
    print("Seeds: 2  |  Epochs: 50")
    print("=" * 60)
    
    results = p.run_phase_a()
    
    print("\n✅ Phase A v2 training complete!")
    
except Exception as e:
    print(f"\n🔴 Training exception: {e}")
    import traceback
    traceback.print_exc()
    print("Partial results are saved on disk.")

## ▶ufe0f Step 6: Push Results to GitHub

In [ ]:
OUT_DIR = "experiments/phase_a/results_v2"
CKPT_DIR = "experiments/phase_a/checkpoints_v2"
cmds = [
    f"git add {OUT_DIR}/*.json {OUT_DIR}/*.csv 2>/dev/null || true",
    f"git add {CKPT_DIR}/*.pt 2>/dev/null || true",
    'git commit -m "Data: Phase A v2 CIFAR-10 D-scaling results from Kaggle" || true',
    'git push origin develop 2>&1 || true'
]

for cmd in cmds:
    res = subprocess.run(cmd, shell=True, capture_output=True, text=True,
                         cwd="/kaggle/working/ThermoRG-NN")
    if res.returncode != 0 and "nothing to commit" not in res.stdout:
        print(f"WARN: {res.stderr[-200:]}")
    else:
        print(f"OK: {cmd[:70]}")

print("\n🎉 Phase A v2 pipeline complete! Results pushed to GitHub.")